### E.3 Lab 6 — Density Matrix Encoding Metrics

### Lab Access and Execution Guide

This guide explains how to run and explore the hands-on quantum computing labs that accompany the book  
**Quantum AI Systems: Theory, Architecture, and Applications** (Professional and Student Volumes).  

The labs are an integral part of the MyQuantumBook project, designed to reinforce key concepts from the chapters through interactive exploration. They are built for execution on **Google Colab** and **IBM Quantum backends** using **Qiskit**, and follow the IEEE-compliant figure, caption, and documentation standards described in the text.  

Each lab is cross-referenced to its corresponding chapter and appendix figure (Appendix E), ensuring reproducibility and scholarly traceability.

**Getting Started**
1. Launch the notebook in Google Colab using the provided badge.
2. Run the setup cells to install Qiskit:
   `!pip install qiskit`

**Using IBM Quantum Systems**
1. Sign up at https://quantum.ibm.com and create an API token.
2. Run the IBMQ setup cell.
3. Replace 'MY_API_TOKEN' with your real token (only needed once).
4. Select backends using `provider.get_backend('ibmq_qasm_simulator')` or others.

**Lab Structure**
Each code section aligns with a chapter from the book.
- Modify and re-run code blocks.
- View circuits with `.draw()`.
- Apply to custom inputs to deepen your understanding.

**Additional Help**
- Refer to the Qiskit Documentation: https://qiskit.org/documentation/
- For support, contact your course instructor or visit the IBM Quantum Community forums.

3. Or launch this lab directly now: [![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jopaneur/QuantumAI-Labs/blob/main/Advanced_Labs/notebooks/Chapter_9_Quantum_Encoding_and_Information_Metrics_for_AI_Advanced_Challenge_Density_Matrix_Simulation_Bloch_Sphere_Shrink_&_Distinguishability.ipynb)


---

**Note for Lab Participants**

Each plot generated in this notebook is automatically saved as a `.png` file under:
Advanced_Labs/figures/


The filenames follow the Appendix E figure numbering (e.g., `E2_1_Bloch_Trajectories.png`, `E2_6_DensityMatrix_Heatmap.png`).  
This allows you to both view results inline in Colab **and** find the corresponding image files for reports, submissions, or cross-references in the book.

**Where Figures Are Saved**
- In **Google Colab**: `/content/Advanced_Labs/figures/`  
- **Locally**: `Advanced_Labs/figures/` (next to your notebook)  
- These images are **not automatically added to GitHub** — commit/push them if you want them in the repo.

**Customizing Save Location**
If you want the figures saved elsewhere, you can change the `subdir` default in the `save_e_figure()` helper or pass a different path each time you call it.


---




---
**Chapter 9 — Quantum Encoding and Information Metrics for AI**

Chapter 9 introduces the density matrix as a powerful formalism for representing both pure and mixed quantum states. It shows how mixtures naturally capture the effects of noise, randomness, and decoherence, and how metrics such as purity, fidelity, and trace distance quantify the robustness of encodings. The chapter emphasizes that encoding fidelity is directly tied to the reliability of quantum AI pipelines: as states contract on the Bloch sphere, they lose information content and become harder to distinguish, challenging downstream learning tasks.

Lab 6 transforms this theoretical framework into a visual and computational experiment. By constructing mixtures of pure and maximally mixed states, learners watch Bloch vectors shrink toward the sphere’s center and see how distinguishability metrics degrade. The exercise reinforces Chapter 9’s theme: geometry and metrics are not abstract — they provide diagnostic handles on how well quantum information survives noise in AI systems.

---

**Advanced Lab 6 — Density Matrix Encoding Metrics**

Challenge: Mixing, Geometry, and Distinguishability

**Goal:** Construct single-qubit density matrices by convexly mixing a pure state with the maximally mixed state. Sweep the mixing probability p, and chart how the Bloch-vector radius, purity, and trace distance to a reference state evolve. This hands-on experiment directly visualizes the chapter’s theme: how noise drives states inward on the Bloch sphere and reduces their distinguishability, illustrating the connection between encoding geometry, operational metrics, and robustness in AI-oriented quantum information processing.
Cross-reference: Appendix E.2, Figure E.2.6.

---

**Task 1 - Helper Utilities**

In [ ]:
# ---- Figure helper (robust; standardized for QAIS labs) ----
import os, matplotlib.pyplot as plt

def save_e_figure(fig_label: str,
                  fig_num: str,
                  lab_id: str = "P2_AdvLab06",
                  subdir: str = "Advanced_Labs/figures",
                  fig=None, ax=None):
    """
    Save the current/explicit figure with IEEE-aligned naming and path.
    Example: save_e_figure("Figure E.2.6a", "a")
             → P2_AdvLab06_E.2.6a.png
    """
    os.makedirs(subdir, exist_ok=True)

    # Get current fig/ax if not supplied
    if fig is None:
        fig = plt.gcf()
    if ax is None:
        ax = fig.axes[0] if fig.axes else None
    if ax is None:
        print("⚠️ No axes found. Draw a plot first, or pass fig/ax explicitly.")
        return

    # Ensure title includes IEEE label
    title = ax.get_title() or ""
    if not title.startswith(fig_label):
        ax.set_title((fig_label + " — " + title).strip(" —"))

    # Construct standardized filename
    fname = f"{lab_id}_{fig_label.replace('Figure ', '').replace('.', '_')}.png"
    outpath = os.path.join(subdir, fname)

    # Save figure
    fig.tight_layout()
    fig.savefig(outpath, dpi=160)

    print(f"✅ Saved {fig_label} as {outpath}")


**Methodology Analysis**

This helper centralizes figure persistence to enforce IEEE-style labeling and consistent file paths across all labs. It creates the target directory (Advanced_Labs/figures) if needed, fetches the current fig and primary ax when not provided, and guards against empty figures by warning if no axes exist. Before saving, it auto-prefixes the axes title with the provided fig_label (for example, “Figure E.2.6a — …”) to keep captions synchronized with Appendix E numbering. Layout is standardized via fig.tight_layout() and export quality via dpi=160, producing reproducible PNGs. Use exactly one call per visual, immediately after the plot is finalized, to avoid duplicate saves and to keep filenames aligned to figure numbers (e.g., P2_AdvLab06_E.2.6a.png). For multi-axes figures, pass the intended axes explicitly with ax= to ensure the correct title prefix.


**Participant Feedback**

Running this cell does not display a plot. When you later save a figure, you should see a console confirmation like:

Saved Advanced_Labs/figures/P2_AdvLab06_E.2.6a.png
If you attempt to save before drawing a plot, you will see:
⚠️ No axes found. Draw a plot first, or pass fig/ax explicitly.
Files are written to Advanced_Labs/figures/ and will overwrite silently if the same name is reused, so expect updated timestamps on re-runs. If your figure contains multiple subplots, pass ax to control which panel’s title is labeled. After saving, you can verify success by checking the folder for the new .png file and confirming the title line on the figure begins with your IEEE label.


---

**Task 2 - Environment Setup and Package Validation**

In [ ]:
# === Environment Setup (Re-runnable; CPU-only backends) ===
import sys, subprocess, pkgutil, os

def ensure(pkg):
    """Install a package if not already available in the environment."""
    if pkg not in {m.name for m in pkgutil.iter_modules()}:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

# Core scientific stack
for pkg in ["numpy", "matplotlib", "scikit-learn", "scipy"]:
    ensure(pkg)

# Quantum SDK
for pkg in ["qiskit", "qiskit-aer"]:
    ensure(pkg)

# Imports
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
try:
    from qiskit_aer import Aer
except ImportError:
    from qiskit.providers.aer import Aer
from qiskit.quantum_info import Statevector, DensityMatrix, SparsePauliOp
from qiskit.visualization import plot_histogram
import qiskit

# Version reporting
print(f"✅ Environment ready — Python {sys.version.split()[0]}")
print("qiskit:", qiskit.__version__)
print("numpy:", np.__version__)
print("matplotlib:", plt.matplotlib.__version__)


**Methodology Analysis**

This environment setup cell ensures a consistent, CPU-only runtime across all learners.
It first defines a lightweight ensure() utility that installs required packages on demand, guaranteeing a reproducible setup even on clean systems such as Google Colab.
It then loads the full scientific and quantum-computing stack — NumPy, Matplotlib, SciPy, Scikit-Learn, and Qiskit (Aer for simulation).
The printout at the end confirms versions, which is crucial for reproducibility and helps diagnose compatibility issues when numerical or visualization discrepancies arise.

This structure also isolates installation logic from analysis logic, a best practice in QAIS labs for ensuring portability between research notebooks and classroom environments.

**Participant Feedback**

When you run this cell:

* You should see confirmation messages such as "Environment ready — Python 3.10.12", followed by version listings for Qiskit and NumPy.

* If any package was missing, it will be quietly installed the first time, after which reruns execute instantly.

No figures or numerical results appear yet; this cell simply prepares the environment. Once this step completes successfully, you can proceed with the quantum encoding and density-matrix metric analyses that follow.


---

**Lab Overview – Density Matrix Encoding and Quantum Information Metrics**

This lab explores how mixed quantum states can represent uncertainty and noise within quantum-AI systems. Learners transition from pure-state representation to density matrices, enabling analysis of partially mixed or decohered states that commonly arise in realistic quantum environments. By comparing Bloch-vector norms, trace distances, and purity trends, participants visualize how statistical mixing alters the geometric and informational structure of qubit states.

**Challenge:**

Construct and analyze a one-qubit family of states that interpolate between pure and maximally mixed regimes. Quantify how the Bloch-vector length and trace distance evolve as the system becomes increasingly mixed, and interpret these changes in terms of information loss and distinguishability.

**Implementation Note:**

The notebook employs Qiskit Quantum Information tools to generate statevectors, convert them into density-matrix form, and compute derived metrics. It uses analytic formulas rather than sampling to ensure deterministic evaluation of fidelity and trace-distance behavior. The environment setup includes helper routines for Bloch-vector extraction and Hermiticity-preserving trace-distance computation, consistent with the broader QAIS benchmarking framework for quantum-state quality assessment.

**Expected Results**

* Figure E.2.6a (Bloch-Vector Norm vs Mixing Parameter p):
The Bloch-vector length decreases smoothly from 1 (pure) to 0 (maximally mixed), illustrating loss of coherence as environmental noise increases.

* Figure E.2.6b (Trace Distance vs Mixing and Angle θ):
The trace distance to a fixed reference state diminishes with both increasing p and angular proximity, forming a gradient surface that encodes state distinguishability in two parameters.

Together, these results demonstrate how density-matrix formalism unifies classical probability and quantum coherence, providing the mathematical foundation for error analysis, quantum memory modeling, and AI-relevant robustness diagnostics in noisy intermediate-scale devices.

**Task 3 - Bloch Vector Shrinkage and Distinguishability Loss (Shared Setup and Computation)**

In [ ]:
# --- Shared setup for Figures E.2.6a and E.2.6b ---


**Methodology Analysis**

This setup cell prepares all the resources needed for Figures E.2.6a and E.2.6b. It defines helper functions to (1) compute Bloch vectors from density matrices, (2) generate single-qubit pure states parametrized by θ, (3) convexly mix states with the maximally mixed state, and (4) calculate trace distance to a reference state. Parameter grids for θ and mixing probability p are established, and arrays are allocated for Bloch norms and trace distances. A reference state is chosen at θ = π/3, and a nested loop fills the arrays by sweeping through mixing levels and state angles. This computation stage ensures that both visualizations can be generated consistently from the same dataset without recalculating values in each figure cell.

**Participant Feedback**

Running this cell will not display any figures, but it sets up the data structures used by the next two figure cells. Expect no plots here, only internal calculations. When complete, variables such as bloch_norm and trace_dist will be populated. If you inspect them, you should see numeric arrays whose values decrease with higher mixing probabilities, reflecting Bloch vector shrinkage and diminishing trace distance. The actual plots will appear in the following figure cells for E.2.6a and E.2.6b.


---

**Task 4 - Bloch vector shrink vs mixing probability**

In [ ]:
# === Figure E.2.6a: Bloch vector shrink vs mixing ===




**Figure E.2.6a. Bloch vector shrink vs mixing probability.**
This figure illustrates how the length of the Bloch vector ‖r‖ decreases as a pure qubit state is convexly mixed with the maximally mixed state. The contraction of the Bloch vector highlights the geometric loss of coherence as the mixing parameter p increases.

**Expected Results**

‖r‖ starts near 1 at p = 0 and decreases steadily toward 0 as p → 1. This reflects that all input states collapse toward the center of the Bloch sphere.

**Technical Analysis (for the visual)**

Mixing follows ρ(p) = (1 − p)ρ_pure + p·I/2, which scales the Bloch vector r by (1 − p). The linear contraction of ‖r‖ demonstrates the loss of state purity under depolarizing-like noise.

**Intuition Sidebar**

Imagine a bright point on the Bloch sphere. Adding noise is like pulling it inward with fog. More noise means less directionality until it disappears at the center.

**Methodology Analysis**

This cell plots the Bloch vector length ‖r‖ as a function of mixing probability p for a representative input angle θ. The calculation uses the array bloch_norm, which was pre-computed in the setup cell by sweeping through all θ and p values. Here, the central column (thetas.size // 2) is selected as the slice for plotting, corresponding to a mid-range θ. The plot visually demonstrates how coherence decreases as more maximally mixed contribution is added, shrinking the Bloch vector radius toward zero.

**Participant Feedback**

Running this cell will display a line plot titled “Bloch vector shrink vs mixing.” You should see the curve start near 1 at p = 0 and steadily decrease toward 0 as p approaches 1. A confirmation message such as
✅ Saved Figure E.2.6a as Advanced_Labs/figures/P2_AdvLab06_E.2.6a.png
will appear in the console, verifying that the figure was stored with the correct filename.


---

**Task 5 - Distinguishability loss vs mixing probability**

In [ ]:
# === Figure E.2.6b: Distinguishability loss vs mixing ===




**Figure E.2.6b. Distinguishability loss vs mixing probability.**
This figure shows how the trace distance between a mixed state and a fixed reference pure state decreases with increasing mixing probability, quantifying the erosion of statistical distinguishability.

**Methodology Analysis**

This cell plots the trace distance D between mixed states ρ(p) and a fixed reference state σ_ref as a function of mixing probability p. The array trace_dist, pre-computed in the setup cell, provides the values by evaluating the trace distance at different θ values and mixing probabilities. Like the previous plot, the central θ slice is chosen for visualization. This plot highlights the operational perspective of noise: as p increases, mixed states become harder to distinguish from the reference, reflecting statistical indistinguishability.

**Participant Feedback**

Running this cell will display a line plot titled “Distinguishability loss vs mixing.” You should see the curve start at a relatively high value at p = 0 (since pure states differ) and then decline as p increases, approaching a smaller value near p = 1. A console message will confirm the save, for example:
✅ Saved Figure E.2.6b as Advanced_Labs/figures/P2_AdvLab06_E.2.6b.png
You can open the saved PNG file to verify the figure matches the plot shown inline.

**Expected Results**

* At p = 0, the distance is large (pure vs pure). As p increases, the distance steadily declines, approaching a minimal value near p = 1.

**Technical Analysis (for the visual)**

* Trace distance D(ρ,σ) = ½‖ρ − σ‖₁ shrinks as ρ contracts toward I/2. The narrowing eigenvalue spectrum reduces distinguishability under measurements.

**Intuition Sidebar**

Think of comparing two coins. Initially, biases differ and you can tell them apart. With mixing, both become more “fair” until they behave nearly identically, erasing your ability to distinguish them.

---
**Wrap-Up for Bloch Vector Shrinkage and Distinguishability Loss**

Figure E.2.6a shows how uniform mixing steadily contracts the Bloch vector radius, providing a geometric picture of coherence loss as states move inward from the surface of the Bloch sphere toward its center. Figure E.2.6b demonstrates how trace distance to a fixed reference state diminishes under the same mixing, quantifying the loss of statistical distinguishability. Together, these results capture the dual impact of noise on quantum states: geometrically, by reducing purity, and operationally, by erasing differences between states. This experiment benchmarks how density matrices serve as a diagnostic lens for robustness of quantum encodings in AI.


**Conclusion**

This lab illustrated how density matrix formalism unifies the description of quantum coherence and its degradation under noise. By mixing pure states with the maximally mixed state, we saw the Bloch vector shrink linearly with mixing probability and trace distance to a reference collapse in parallel. These results emphasize how both geometric and operational metrics reveal the same story: noise drives states toward indistinguishability, reducing their usefulness as reliable encodings. In the context of Quantum AI Systems, these diagnostics are essential for evaluating the resilience of encodings against decoherence and randomness.

**Key Take-Aways**

Uniform mixing scales the Bloch vector radius by (1 − p), shrinking state purity.

* Trace distance decreases with mixing, quantifying reduced distinguishability between states.

* Density matrices capture both geometry (Bloch sphere contraction) and information-theoretic metrics in one framework.

* Noise transforms distinguishable pure states into nearly indistinguishable mixed states.

Encoding metrics like Bloch radius and trace distance provide actionable benchmarks for noise sensitivity in quantum AI pipelines.

**Congratulations**

Excellent work! You successfully implemented density-matrix simulations, visualized Bloch vector contraction, and quantified distinguishability loss. By linking geometry and trace distance, you now understand how noise undermines both the purity and information content of quantum states. These are core skills for analyzing and designing robust encodings in Quantum AI Systems.

### Appendix E → Appendix B Cross-Reference
See **Appendix B — Quick Self-Check, Chapter 9 — Quantum Encoding and Information Metrics for AI**:  
- Questions 3 and 4 (coherence loss and trace-distance metrics).  
They relate to the coherence-decay and robustness diagnostics demonstrated in **E.3 Lab 6** (Figures E.3.6a–b).


---

**How to save or submit your work**

- **If you are a student (graded/evaluated):**  
  1. Export your key plots or the entire notebook to PDF (File → Print/Save as PDF).  
  2. Save the notebook (`.ipynb`).  
  3. Bundle any extra files (CSVs/images) if used.  
  4. Upload to your LMS or repository as instructed (include your name and lab number).  
  5. Repro checklist: set a random seed where applicable, note backend and shots, and list package versions.  

- **If you are a professional/self‑learner (non‑graded exercise):**  
  1. Save the notebook (`File → Download .ipynb`) to your computer for personal reference.  
  2. Optionally export to PDF for archiving.  
  3. Keep any generated plots or data locally.  
  4. Use version control (GitHub, GitLab) if you wish to track your personal progress.



---
